# RaceEngineer — train XGBoost Ranker chọn vòng pit

Notebook cho Colab hoặc Jupyter Python 3.11/3.12. Chạy từ trên xuống.
**Đầu ra là vòng pit được chọn**, không phải model chỉ dự đoán pace.
Một query = trạng thái của một xe tại một thời điểm; mỗi row = một vòng pit ứng viên.
Model xếp hạng ứng viên; core lọc luật/fuel rồi lấy hạng cao nhất.

**Hiện repo chưa có dataset pit được gán nhãn.** Mặc định `MODE="demo"` tạo dữ liệu
giả để thực hành toàn bộ pipeline. Điểm demo chỉ đo khả năng học quy luật giả;
không chứng minh chiến thuật đua tốt. Artifact luôn xuất `deployment_ready=false`.
Đổi sang `real` khi có candidate CSV đúng hợp đồng ở dưới. Notebook không tự thu
telemetry từ game và không tự biến timing logs thành nhãn tối ưu.

Phạm vi v1: cuộc đua tính bằng vòng, còn đúng một stop bắt buộc, một cấu hình dịch vụ
pit cố định, điều kiện khô và race profile được hỗ trợ. Không áp dụng cho no-stop,
multi-stop hoặc race tính giờ cho đến khi mở rộng action/schema và đánh giá riêng.
Vòng pit k nghĩa là vào pit **cuối vòng k**; current_lap là vòng đang chạy (1-based).


## 1. Dữ liệu: ưu tiên đúng domain, không trộn mặc định

| Ưu tiên | Nguồn | Cách sử dụng / giới hạn |
|---|---|---|
| 1 | AC/ACC: tự ghi RaceState/RaceHistory, pit events và race profile | Nguồn chính. Ghi từng vòng/sự kiện; log fuel, tuổi lốp nếu biết, pace, gap, service. Recorder của app nằm trong plan, chưa có ở notebook. |
| 1 | Log sim của bạn/league cho phép sử dụng; ACC qua công cụ telemetry | Chuẩn hóa đơn vị, simulator/version, track/layout, car/class, luật race. File MoTeC/AiM cần export/adapter theo format thật; notebook không đọc binary .ld/.drk. |
| 2 | GT3: SRO/GT World Challenge timing chính thức | Timing/pit history hỗ trợ phân tích và hiệu chỉnh; không mặc định có fuel hoặc tyre wear. |
| 2 | WEC: Al Kamel, ưu tiên LMGT3; tách Hypercar/LMP2/GTE | Multiclass, driver changes, rules khác ACC; cần profile riêng. |
| 2 | F1: FastF1 | Có cell tải từng session tùy chọn. Dữ liệu phụ; không gán F1 thành GT3. |
| 3 | GT4, touring, one-make hoặc hạng thấp hơn có timing chính thức | Chỉ thêm sau khi nguồn ưu tiên đã được đánh giá; không giả định hạng thấp có luật giống GT3. |

Không có dataset nguồn nào ở đây được khẳng định có sẵn nhãn **pit tối ưu**.
Đọc quyền sử dụng từng nguồn trước khi tải/train/chia sẻ; Al Kamel nêu hạn chế phân phối
dữ liệu. Không commit raw data hoặc model vào Git. Lưu URL, ngày lấy và điều kiện sử dụng
trong provenance. Xác nhận quyền train và quyền phân phối model riêng khi cần.

Nguồn tham khảo (kiểm tra 2026-09-24):
- [AiM: ghi telemetry ACC](https://www.aimsportsystems.com.au/download/doc/eng/simracing/AssettoCorsaCompetizione_100_eng.pdf)
- [SRO GT World Challenge results](https://www.gt-world-challenge-europe.com/results)
- [FIA WEC / Al Kamel](https://fiawec.alkamelsystems.com/)
- [FastF1](https://github.com/theOehrly/Fast-F1)
- [TUM: mô phỏng chiến thuật theo vòng](https://github.com/TUMFTM/race-simulation)
- [XGBoost learning-to-rank](https://xgboost.readthedocs.io/en/stable/tutorials/learning_to_rank.html)
- [XGBoost model IO](https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html)


### F1 public-data pilot (not for deployment)

Source: [Mendez and Thraves F1 pit-stop research](https://github.com/Felipe-mendezp/f1-pit-stop-drl), 2021-2024 CSVs under the repository MIT license. From the repository root, run `python training/pit_strategy/prepare_f1_pilot.py` to create candidates.

For the pilot set `MODE="pilot"`, `DATA_PATH=Path(".local_train/f1_pit_candidates.csv")`, `TARGET_PROFILE="f1_dry_one_stop_pilot"`, `ALLOWED_DOMAINS={"f1"}`. Labels come from a simple linear stint simulator with no traffic, weather, fuel or full pit rules. The source lacks exact session dates, so split order uses source event order within each year; it is not a verified within-season chronological split. This is not validated optimal pit timing. Manifest keeps `deployment_ready=false`; never use this F1 model for AC/ACC.


In [ ]:
# Chạy cell này để cài môi trường; cần internet. Không tự tải dataset/model lớn.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                      'xgboost>=3.0,<4', 'pandas>=2.2,<3', 'numpy>=1.26,<3'])


In [ ]:
from pathlib import Path
import hashlib, json, platform, time
import numpy as np
import pandas as pd
import xgboost as xgb

MODE = 'demo'  # 'real' để đọc candidate CSV đã gán nhãn
DATA_PATH = Path('pit_candidates.csv')
OUT = Path('pit_ranker_output')
OUT.mkdir(exist_ok=True)
SEED = 42
# Model v1 huấn luyện cho MỘT track/car/rules profile. Mở rộng sau khi có bằng chứng.
TARGET_PROFILE = 'acc_gt3_example_dry_one_stop_lap_race'
ALLOWED_DOMAINS = {'sim_acc', 'sim_ac'}
# Không thêm gt3/wec/f1/lower chỉ để tăng số row. Làm thí nghiệm riêng,
# cùng test set sim cố định và kiểm tra negative transfer trước.
FEATURES = [
    'laps_remaining', 'fuel_laps_remaining', 'reserve_laps', 'stint_laps',
    'pace_mean_s', 'pace_trend_s', 'gap_ahead_s', 'gap_behind_s',
    'pit_loss_s', 'window_open_offset', 'window_close_offset', 'candidate_offset',
]
REQUIRED_FEATURES = [f for f in FEATURES if f not in {'gap_ahead_s', 'gap_behind_s'}]
print('Versions:', platform.python_version(), xgb.__version__, pd.__version__, np.__version__)


## 2. Tải/import timing thô (tùy chọn, tách khỏi training)

Sim: xuất CSV có UTC session start/end, ID event/session/driver, lap, lap time (s),
fuel (L), pit in/out, tyre-set ID hoặc tuổi lốp nếu nguồn xác nhận; đánh dấu lap invalid,
traffic/cờ/hư hại. Thiếu field để trống, không điền 0 như một phép đo.
Giữ raw logs bất biến; adapter cần audit tên cột, đơn vị và thời điểm field khả dụng.

Cell FastF1 dưới đây chỉ tải **một race** khi bật cờ. Lưu các lap đã quan sát;
PitInTime là kết quả/hành động trong lịch sử, không phải nhãn pit tối ưu.
Full lap time của vòng đang chạy, pit tương lai và thứ hạng cuối race không được làm feature.


In [ ]:
DOWNLOAD_F1 = False
if DOWNLOAD_F1:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'fastf1>=3.6,<4'])
    import fastf1
    cache = OUT / 'fastf1_cache'
    cache.mkdir(exist_ok=True)
    fastf1.Cache.enable_cache(str(cache))
    session = fastf1.get_session(2024, 'Bahrain', 'R')  # thay đúng event muốn dùng
    session.load(telemetry=False, weather=False, messages=False)
    laps = session.laps.copy()
    raw = pd.DataFrame({
        'driver_id': laps['DriverNumber'], 'lap': laps['LapNumber'],
        'lap_time_s': laps['LapTime'].dt.total_seconds(),
        'pit_in_time_s': laps['PitInTime'].dt.total_seconds(),
        'pit_out_time_s': laps['PitOutTime'].dt.total_seconds(),
        'stint': laps['Stint'], 'tyre_age_laps': laps['TyreLife'],
        'compound': laps['Compound'], 'track_status': laps['TrackStatus'],
    })
    raw.to_csv(OUT / 'f1_2024_bahrain_observed_laps.csv', index=False)
    print('Raw timing only; chưa dùng để train ranker:', len(raw))

# CSV timing chính thức GT3/WEC hoặc CSV export sim: kiểm tra schema trước.
# Không có parser chung đáng tin cho mọi mùa/nhà cung cấp.
RAW_CSV = None  # ví dụ Path('/content/23_Analysis_Race.CSV')
RAW_SEPARATOR = ';'  # đặt theo file thật
if RAW_CSV is not None:
    raw_timing = pd.read_csv(RAW_CSV, sep=RAW_SEPARATOR, encoding='utf-8-sig')
    print(raw_timing.columns.tolist())
    display(raw_timing.head())


In [ ]:
# RaceEngineer ghi lap v? pit event d??i d?ng JSON Lines ? Qt AppLocalDataLocation.
# Tr? RAW_SIM_JSONL v?o b?n sao file t? m?y ch?i game ?? ki?m tra v? chu?n h?a.
RAW_SIM_JSONL = None  # v? d? Path('pit_strategy_laps.jsonl')
if RAW_SIM_JSONL is not None:
    sim_observed = pd.read_json(RAW_SIM_JSONL, lines=True)
    print(sim_observed.groupby(['simulator', 'record_type']).size())
    display(sim_observed.head())
    # Raw observations kh?ng c? outcome c?a c?c v?ng pit ch?a ch?n.
    # D?ng ch?ng ?? hi?u ch?nh simulator/g?n nh?n candidate, kh?ng fit Ranker tr?c ti?p.


## 3. Hợp đồng candidate CSV và cách gán nhãn

Metadata bắt buộc:
`session_id, session_start_utc, session_end_utc, decision_id, source_domain,
profile_id, current_lap, candidate_lap, label_method, label_version, provenance,
relevance, outcome_cost_s` + 12 feature trong `FEATURES`.

- `session_id`: ID toàn race/event, dùng chung cho mọi xe để không rò rỉ giữa train/test.
  `decision_id`: duy nhất trong session, bao gồm driver + thời điểm quyết định.
- `laps_remaining`: số vòng từ current_lap đến đích, gồm vòng đang chạy.
- `fuel_laps_remaining`: số vòng tương đương còn chạy được từ vị trí hiện tại.
  V1 lọc fuel bảo thủ bằng `candidate_offset + 1 + reserve_laps <= fuel_laps_remaining`.
  Giới hạn này có thể loại phương án sát fuel; runtime không được dùng phép lọc lỏng hơn.
- `stint_laps`: tuổi bộ lốp xác nhận; pit không thay lốp không reset tuổi lốp.
- `pace_mean_s`, `pace_trend_s`: chỉ tính từ các vòng đã hoàn thành trước decision.
  `pace_trend_s > 0` nghĩa chậm đi. Gap không có => NaN. `pit_loss_s` là ước tính
  có trước decision, không dùng thời gian pit thực tế ở tương lai.
- Window offsets là luật pit đã biết chuyển sang vòng cho race tính vòng; nếu không có
  giới hạn riêng, dùng [0, laps_remaining - 1]. Window theo thời gian chưa hỗ trợ v1.
- Candidate offset = candidate_lap - current_lap; liệt kê ĐẦY ĐỦ các vòng hợp lệ.
- `source_domain`: sim_acc/sim_ac/gt3/wec/f1/lower. `profile_id` phân biệt simulator,
  car, track/layout, rules, service, weather; không trộn profile khác trong model v1.
- Nhãn `relevance`: số nguyên 0..3, cao hơn tốt hơn; chỉ so sánh trong cùng decision.
- `outcome_cost_s`: expected remaining race time trong cùng giả định của mô phỏng;
  chỉ dùng gán nhãn/đánh giá, TUYỆT ĐỐI không đưa vào feature. Có thể NaN khi nhãn expert.

**Cách làm nhãn thật:** hiệu chỉnh simulator trên TRAIN sessions; với mỗi state, thử
tất cả vòng pit trên cùng tập kịch bản tương lai (cùng random seeds), cùng dịch vụ pit
và cùng chiến thuật sau stop. Loại vi phạm luật/fuel trước khi xếp hạng. Ví dụ grades:
regret <= 0.5s → 3; <= 2s → 2; <= 5s → 1; còn lại → 0. Đây là ngưỡng khởi đầu,
cần hiệu chỉnh theo profile, không phải tiêu chuẩn chung. Cùng grade nghĩa gần tương đương.
Ghi phiên bản simulator và nguồn calibration vào `label_version`/`provenance`.

Expert có thể xếp hạng với `label_method=expert_review`; không giả tạo outcome cost.
Mô phỏng đã hiệu chỉnh dùng `calibrated_simulation`. Pit đã quan sát chỉ dùng cho
behavior cloning riêng; notebook này từ chối coi đó là optimal label.
Test scenarios phải độc lập; không dùng kết quả test để tune simulator hoặc thresholds.
Notebook cung cấp toy labeler để học pipeline, **không cung cấp simulator đã hiệu chỉnh**.


In [ ]:
def demo_candidates():
    rng = np.random.default_rng(SEED)
    rows = []
    for event in range(40):
        start = pd.Timestamp('2025-01-01', tz='UTC') + pd.Timedelta(days=event)
        for decision in range(8):
            lap = 8 + decision
            state = dict(laps_remaining=30-lap, fuel_laps_remaining=float(rng.uniform(4, 9)),
                         reserve_laps=0.5, stint_laps=float(lap),
                         pace_mean_s=float(rng.uniform(95, 110)),
                         pace_trend_s=float(rng.uniform(-0.1, 0.6)),
                         gap_ahead_s=float(rng.uniform(0.5, 15)),
                         gap_behind_s=float(rng.uniform(0.5, 15)),
                         pit_loss_s=float(rng.uniform(20, 35)),
                         window_open_offset=0, window_close_offset=5)
            upper = min(5, int(np.floor(state['fuel_laps_remaining']-1.5)))
            # Quy luật GIẢ: không phải physics, không dùng để quyết định pit thật.
            preferred = np.clip(4 - 5*state['pace_trend_s'] + state['gap_behind_s']/15, 0, upper)
            costs = [1000 + 2*(offset-preferred)**2 for offset in range(upper+1)]
            for offset, cost in enumerate(costs):
                regret = cost-min(costs)
                rows.append(dict(**state, candidate_offset=offset,
                    session_id=f'demo_{event:03d}', decision_id=f'car1_{decision}',
                    session_start_utc=start.isoformat(),
                    session_end_utc=(start+pd.Timedelta(hours=1)).isoformat(),
                    source_domain='demo', profile_id=TARGET_PROFILE, current_lap=lap,
                    candidate_lap=lap+offset, relevance=3 if regret<=0.5 else 2 if regret<=2 else 1 if regret<=5 else 0,
                    outcome_cost_s=cost, label_method='toy_simulation', label_version='toy_v1',
                    provenance='generated_demo_not_racing_evidence'))
    return pd.DataFrame(rows)

if MODE not in {'demo', 'real', 'pilot'}:
    raise ValueError('MODE phải là demo hoặc real')
if MODE == 'demo':
    data = demo_candidates()
else:
    if not DATA_PATH.is_file():
        raise FileNotFoundError(f'Cần candidate CSV đã gán nhãn: {DATA_PATH.resolve()}')
    data = pd.read_csv(DATA_PATH)
    required_selection = {'source_domain', 'profile_id'}
    if not required_selection.issubset(data.columns):
        raise ValueError('CSV thiếu source_domain/profile_id')
    data = data[data.source_domain.isin(ALLOWED_DOMAINS) & data.profile_id.eq(TARGET_PROFILE)].copy()
    if data.empty:
        raise ValueError('Không có dữ liệu đúng domain/profile. Không tự fallback sang dữ liệu giả.')
print('Mode:', MODE, 'Rows:', len(data))


In [ ]:
META = ['session_id', 'session_start_utc', 'session_end_utc', 'decision_id', 'source_domain',
        'profile_id', 'current_lap', 'candidate_lap', 'label_method', 'label_version',
        'provenance', 'relevance', 'outcome_cost_s']
pd.DataFrame(columns=META + FEATURES).to_csv(OUT/'candidate_template.csv', index=False)

def validate(frame, mode):
    d = frame.copy()
    missing = set(META + FEATURES) - set(d.columns)
    if missing:
        raise ValueError(f'Thiếu cột: {sorted(missing)}')
    for col in META:
        if col != 'outcome_cost_s' and (d[col].isna().any() or d[col].astype(str).str.strip().eq('').any()):
            raise ValueError(f'Metadata rỗng: {col}')
    for col in FEATURES + ['current_lap', 'candidate_lap', 'relevance', 'outcome_cost_s']:
        d[col] = pd.to_numeric(d[col], errors='raise')
    numeric = d[FEATURES + ['outcome_cost_s']].to_numpy(dtype=float)
    if np.isinf(numeric).any() or d[REQUIRED_FEATURES].isna().any().any():
        raise ValueError('Feature bắt buộc thiếu hoặc có infinity')
    for col in ['current_lap', 'candidate_lap', 'laps_remaining', 'candidate_offset',
                'window_open_offset', 'window_close_offset', 'relevance']:
        if not np.isfinite(d[col]).all() or not d[col].eq(np.floor(d[col])).all():
            raise ValueError(f'Cần số nguyên hữu hạn: {col}')
    if not d.relevance.between(0, 3).all():
        raise ValueError('relevance phải trong 0..3')
    if not ((d.current_lap >= 1) & (d.laps_remaining >= 1) & (d.reserve_laps >= 0)
            & (d.stint_laps >= 0) & (d.pace_mean_s > 0) & (d.pit_loss_s >= 0)).all():
        raise ValueError('Giá trị vật lý không hợp lệ')
    for col in ['gap_ahead_s', 'gap_behind_s']:
        if (d[col].dropna() < 0).any():
            raise ValueError(f'Gap âm: {col}')
    if not d.candidate_lap.eq(d.current_lap + d.candidate_offset).all():
        raise ValueError('Sai quy ước current_lap/candidate_offset')
    keys = ['session_id', 'decision_id']
    if d.duplicated(keys + ['candidate_lap']).any():
        raise ValueError('Ứng viên bị trùng')
    for col in ['session_start_utc', 'session_end_utc']:
        d[col] = pd.to_datetime(d[col], utc=True, errors='raise')
    if not (d.session_end_utc > d.session_start_utc).all():
        raise ValueError('Sai thời gian session')
    for col in ['session_start_utc', 'session_end_utc', 'source_domain', 'profile_id']:
        if d.groupby('session_id')[col].nunique(dropna=False).gt(1).any():
            raise ValueError(f'Metadata session không đồng nhất: {col}')
    state_cols = [f for f in FEATURES if f != 'candidate_offset'] + ['current_lap', 'label_method', 'label_version']
    for _, g in d.groupby(keys, sort=False):
        if g[state_cols].nunique(dropna=False).gt(1).any():
            raise ValueError('Feature trạng thái phải giống nhau trong một decision')
        row = g.iloc[0]
        lower = max(0, int(row.window_open_offset))
        upper = min(int(row.window_close_offset), int(row.laps_remaining)-1,
                    int(np.floor(row.fuel_laps_remaining-row.reserve_laps-1)))
        if upper < lower or set(g.candidate_offset) != set(range(lower, upper+1)):
            raise ValueError('Ứng viên thiếu hoặc vi phạm fuel/window/race end')
        if row.label_method in {'toy_simulation', 'calibrated_simulation', 'research_simulation'}:
            if g.outcome_cost_s.isna().any() or (g.outcome_cost_s < 0).any():
                raise ValueError('Simulation label cần outcome cost hợp lệ')
            ordered = g.sort_values('outcome_cost_s')
            if (ordered.relevance.diff().dropna() > 0).any():
                raise ValueError('Grade mâu thuẫn outcome cost')
    allowed = {'toy_simulation'} if mode == 'demo' else ({'research_simulation'} if mode == 'pilot' else {'calibrated_simulation', 'expert_review'})
    if not set(d.label_method).issubset(allowed):
        raise ValueError('Loại nhãn không được phép; observed pit không phải optimal label')
    if mode in {'real', 'pilot'} and not set(d.source_domain).issubset(ALLOWED_DOMAINS):
        raise ValueError('Domain ngoài cấu hình')
    return d.sort_values(keys + ['candidate_offset']).reset_index(drop=True)

data = validate(data, MODE)
# Kiểm tra nhỏ cho data contract, không train model.
bad = data.copy()
bad.loc[0, 'candidate_lap'] += 1
try:
    validate(bad, MODE)
except ValueError:
    pass
else:
    raise AssertionError('Validator không phát hiện sai candidate_lap')
print(data.groupby(['source_domain', 'label_method']).size())


## 4. Chia theo event và thời gian; giữ test độc lập

Không random split các lap/row. Toàn bộ xe, decision và bản sao scenario của cùng
event phải chung session_id. Cần ít nhất 10 event ở đây để chạy pipeline (không phải
cam kết đủ dữ liệu thực tế). Nếu event chồng thời gian qua ranh giới split, dừng để
gom event hoặc đặt split thủ công. Track/car khác profile cần model/evaluation riêng.


In [ ]:
sessions = data[['session_id', 'session_start_utc', 'session_end_utc']].drop_duplicates().sort_values('session_start_utc')
if len(sessions) < 10:
    raise ValueError('Cần ít nhất 10 event độc lập cho train/validation/test demo pipeline')
a, b = int(len(sessions)*0.6), int(len(sessions)*0.8)
parts = [sessions.iloc[:a], sessions.iloc[a:b], sessions.iloc[b:]]
for left, right in zip(parts, parts[1:]):
    if left.session_end_utc.max() >= right.session_start_utc.min():
        raise ValueError('Event chồng thời gian qua split; cần regroup/purge thủ công')
train, valid, test = [data[data.session_id.isin(p.session_id)].copy() for p in parts]
if not train.groupby(['session_id', 'decision_id']).relevance.nunique().gt(1).any():
    raise ValueError('Train không có quyết định với thứ hạng khác nhau để học')
for name, part in zip(['train', 'valid', 'test'], [train, valid, test]):
    print(name, len(part), 'rows;', part.session_id.nunique(), 'events')

def matrix(part):
    # Row thứ tự cùng qid liên tiếp; không đưa metadata/labels vào X.
    part = part.sort_values(['session_id', 'decision_id', 'candidate_offset']).reset_index(drop=True)
    qid = part.groupby(['session_id', 'decision_id'], sort=False).ngroup().to_numpy()
    return part, part[FEATURES].astype('float32'), part.relevance.to_numpy(), qid

train, Xtr, ytr, qtr = matrix(train)
valid, Xva, yva, qva = matrix(valid)
test, Xte, yte, qte = matrix(test)


## 5. Train trên CPU, một luồng

Điểm khởi đầu nhỏ: tối đa 96 cây, depth 3, early stopping trên validation.
Không grid search tự động. Tăng độ lớn chỉ khi validation và đo runtime chứng minh cần.
Model nhận feature số và trực tiếp xếp hạng vòng pit; không cần LAYA/LLM.


In [ ]:
ranker = xgb.XGBRanker(
    objective='rank:pairwise', tree_method='hist', device='cpu', n_jobs=1,
    n_estimators=96, max_depth=3, learning_rate=0.06,
    min_child_weight=2, reg_lambda=2, random_state=SEED,
    eval_metric='ndcg@1', early_stopping_rounds=12,
)
ranker.fit(Xtr, ytr, qid=qtr, eval_set=[(Xva, yva)], eval_qid=[qva], verbose=False)
# Cắt cây sau best iteration để C API không vô tình dùng thêm cây khác Python.
booster = ranker.get_booster()[:ranker.best_iteration + 1]
booster.set_param({'nthread': 1, 'device': 'cpu'})
print('Selected trees:', booster.num_boosted_rounds())


In [ ]:
def evaluate(part, scores):
    d = part.copy()
    d['score'] = scores
    results = []
    for _, g in d.groupby(['session_id', 'decision_id'], sort=False):
        # Tie-break deterministic: pit sớm hơn khi điểm bằng nhau.
        chosen = g.sort_values(['score', 'candidate_offset'], ascending=[False, True]).iloc[0]
        # Baseline đơn giản: vòng giữa cửa sổ hợp lệ, tie chọn sớm.
        middle = (g.candidate_offset.min() + g.candidate_offset.max()) / 2
        baseline = g.iloc[np.argmin(np.abs(g.candidate_offset.to_numpy()-middle))]
        earliest = g.loc[g.candidate_offset.idxmin()]
        max_grade = g.relevance.max()
        item = dict(domain=chosen.source_domain,
                    top_grade_hit=float(chosen.relevance == max_grade),
                    baseline_top_grade_hit=float(baseline.relevance == max_grade),
            earliest_top_grade_hit=float(earliest.relevance == max_grade),
                    ndcg_at_1=float((2**chosen.relevance-1)/(2**max_grade-1)) if max_grade else 1.0)
        # Regret chỉ có nghĩa trong simulation tạo nhãn; không phải causal proof ngoài đời.
        if chosen.label_method in {'toy_simulation', 'calibrated_simulation', 'research_simulation'}:
            item['simulation_regret_s'] = float(chosen.outcome_cost_s-g.outcome_cost_s.min())
            item['baseline_simulation_regret_s'] = float(baseline.outcome_cost_s-g.outcome_cost_s.min())
            item['earliest_simulation_regret_s'] = float(earliest.outcome_cost_s-g.outcome_cost_s.min())
        results.append(item)
    return pd.DataFrame(results)

test_matrix = xgb.DMatrix(Xte, feature_names=FEATURES)
scores = booster.predict(test_matrix)
evaluation = evaluate(test, scores)
print(evaluation.groupby('domain').mean(numeric_only=True))
first = test[test.session_id.eq(test.iloc[0].session_id) & test.decision_id.eq(test.iloc[0].decision_id)].copy()
first_scores = booster.predict(xgb.DMatrix(first[FEATURES].astype('float32'), feature_names=FEATURES))
chosen_lap = int(first.iloc[np.argmax(first_scores)].candidate_lap)
print('Ví dụ output module: pit cuối vòng', chosen_lap)
print('Ranking score không phải xác suất thắng hoặc confidence đã hiệu chỉnh.')


## 6. Xuất model, schema, provenance và vector đối chiếu C++

`pit_ranker.json` dùng XGBoost native C API; không pickle, không cần Python runtime trong app.
`feature_schema.json` giữ thứ tự float32/NaN và quy ước vòng. JSON null trong vector test
phải chuyển thành NaN trước khi đưa vào C API. Không scale khác giữa Python/C++.

Artifact được xuất để review, không tự cài vào app. Manifest khóa deployment: cần kiểm tra
real sim holdout, so sánh baseline, live shadow mode, C++ score parity và perf khi game chạy.
Model demo không bao giờ được đổi cờ để phát hành. Ranker chưa hỗ trợ cả AC/ACC toàn bộ
track chỉ vì đã train thành công một profile.


In [ ]:
model_path = OUT / 'pit_ranker.json'
booster.save_model(model_path)
restored = xgb.Booster()
restored.load_model(model_path)
restored.set_param({'nthread': 1, 'device': 'cpu'})
np.testing.assert_allclose(restored.predict(test_matrix), scores, rtol=1e-6, atol=1e-6)

def write_json(name, obj):
    (OUT/name).write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

write_json('feature_schema.json', dict(version=1, features=FEATURES, dtype='float32',
    missing='NaN', candidate='pit at end of current_lap + candidate_offset',
    required=REQUIRED_FEATURES, profile_id=TARGET_PROFILE, tie_break='earliest_candidate',
    scope='dry, lap-count race, exactly one required remaining stop, fixed service'))
vectors = first[FEATURES].astype(object).where(first[FEATURES].notna(), None).values.tolist()
write_json('parity_vectors.json', dict(features=FEATURES, rows=vectors,
    expected_scores=first_scores.astype(float).tolist(),
    candidate_laps=first.candidate_lap.astype(int).tolist(), selected_lap=chosen_lap,
    atol=1e-6, rtol=1e-6))
split_ids = {n: p.session_id.tolist() for n, p in zip(['train','validation','test'], parts)}
write_json('split_sessions.json', split_ids)
evaluation.to_csv(OUT/'evaluation_per_decision.csv', index=False)
data[META].drop_duplicates().to_csv(OUT/'label_provenance.csv', index=False)
canonical = data.to_csv(index=False).encode('utf-8')
write_json('manifest.json', dict(
    schema_version=1, mode=MODE, deployment_ready=False,
    blocked_by=['demo_data' if MODE=='demo' else ('research_simulation_only' if MODE=='pilot' else 'pending_independent_validation'),
                'pending_cpp_parity', 'pending_in_game_cpu_frametime_measurement'],
    model_sha256=hashlib.sha256(model_path.read_bytes()).hexdigest(),
    dataset_sha256=hashlib.sha256(canonical).hexdigest(),
    profile_id=TARGET_PROFILE, domains=sorted(data.source_domain.unique().tolist()),
    label_methods=sorted(data.label_method.unique().tolist()),
    versions=dict(python=platform.python_version(), xgboost=xgb.__version__, pandas=pd.__version__, numpy=np.__version__),
    trees=booster.num_boosted_rounds(), objective='rank:pairwise', nthread=1,
    metrics=evaluation.drop(columns='domain').mean().to_dict(),
))
# Microbenchmark tùy chọn chỉ đo inference đã warm, không đại diện CPU%/FPS trong game.
RUN_MICROBENCHMARK = False
if RUN_MICROBENCHMARK:
    batch = xgb.DMatrix(first[FEATURES].astype('float32'), feature_names=FEATURES)
    for _ in range(5):
        restored.predict(batch)
    samples = []
    cpu_start = time.process_time()
    for _ in range(100):
        begin = time.perf_counter()
        restored.predict(batch)
        samples.append((time.perf_counter()-begin)*1000)
    print('Warm batch wall p50/p95 ms:', np.percentile(samples, [50,95]))
    print('CPU ms/call:', (time.process_time()-cpu_start)*1000/100)
print('Artifacts:', OUT.resolve(), '— deployment remains disabled')


## 7. Đưa model vào RaceEngineer — checklist thực hiện theo plan.md mục 13

1. Recorder + race profile + candidate builder dùng cùng feature contract.
2. Native C API, `nthread=1`, load một lần, worker riêng, một batch cho các ứng viên.
3. So vector parity, kiểm tra hash/schema/profile, thiếu dữ liệu thì báo insufficient data.
4. Model chọn vòng; C++ chỉ lọc tính hợp lệ, kiểm tra freshness và dispatch radio.
5. Tính lại theo lap hoặc sự kiện quan trọng; theo dõi nhẹ liên tục để gọi pit trước điểm rẽ.
   Reset recommendation/callouts khi đổi session, pit xong, disconnect hoặc tắt strategy.
6. Đánh giá theo session: top-grade hit, NDCG, simulation regret và baseline; live shadow
   dùng kiểm tra tính khả thi/stability, không tuyên bố counterfactual optimum từ replay.
7. Đo CPU ms/call, calls/min, RAM và game frametime với strategy off/on. Không dùng latency
   của notebook để hứa phần trăm CPU trên máy người dùng.

Không phát hành khi chỉ có demo, khi schema khác runtime hoặc model chưa được kiểm chứng.
Nếu không có dataset thật, dừng ở pipeline và recorder; không đổi tên toy labels thành real.


## ??ng g?i sau khi qua c?c gate

Runtime c?n `pit_ranker.json`, `feature_schema.json`, `parity_vectors.json`,
`manifest.json`, `profile.json`, `xgboost.dll` v? `LICENSE` trong
`models/pit_strategy/`. D?ng `production/config/pit_strategy.profile.example.json`
?? ?i?n ch?nh x?c simulator, track, car, lu?t pit v? race length. DLL l?y t?
b?n XGBoost c?ng phi?n b?n ?? train, k?m license. Ghi SHA-256 c?a DLL v?o
`manifest.xgboost_sha256`. Ch? ??t `deployment_ready=true` sau khi model th?t
??t held-out sim evaluation, C++ parity, shadow mode v? ?o CPU/frametime khi game ch?y.
CMake ch? package bundle ???c duy?t; app ki?m tra checksum, schema v? profile.
Model demo tuy?t ??i kh?ng ???c promote.
